In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No GPU allocated. Runtime -> Change runtime type -> T4 GPU, then Run all again.')

print('GPU   :', torch.cuda.get_device_name(0))
print('torch :', torch.__version__)

GPU   : Tesla T4
torch : 2.11.0+cu130


In [ ]:
import glob
import os

from google.colab import drive
drive.mount('/content/drive')

MYDRIVE = '/content/drive/MyDrive'
WORK    = '/content/work'

DS1 = f'{MYDRIVE}/ecg-paper/artifacts/ds1.npz'
if not os.path.exists(DS1):
    print('not at the expected path, searching your Drive...')
    hits = glob.glob(f'{MYDRIVE}/**/ds1.npz', recursive=True)
    assert hits, 'ds1.npz not found anywhere in your Drive -- ask Rishabh to send it'
    DS1 = hits[0]

print('ds1.npz :', DS1)

!rm -rf {WORK}
!mkdir -p {WORK}/src {WORK}/artifacts
!cp "{DS1}" {WORK}/artifacts/ds1.npz

%cd {WORK}
!ls -la artifacts/

Mounted at /content/drive
ds1.npz : /content/drive/MyDrive/ecg-paper/artifacts/ds1.npz
/content/work
total 12068
drwxr-xr-x 2 root root     4096 Oct 11 17:06 .
drwxr-xr-x 4 root root     4096 Oct 11 17:06 ..
-rw------- 1 root root 12346552 Oct 11 17:06 ds1.npz


In [ ]:
# torch and sklearn ship with Colab; these two do not.
!pip install -q xgboost imbalanced-learn

In [ ]:
import os
import shutil

from google.colab import files

NEEDED = {'config.py', 'imbalance.py', 'models.py', 'run_grid.py',
          'train_cnn.py', 'folds.json'}

print('Select all six files in one go:')
print('   ', '  '.join(sorted(NEEDED)))
print()

uploaded = files.upload()

os.makedirs(f'{WORK}/src', exist_ok=True)
for name in uploaded:
    shutil.move(name, f'{WORK}/src/{name}')

got = {f for f in os.listdir(f'{WORK}/src')
       if f.endswith(('.py', '.json'))}
missing = NEEDED - got
extra = got - NEEDED

print()
for f in sorted(got):
    print(f'  {f:18} {os.path.getsize(f"{WORK}/src/{f}"):>7,} bytes')

if extra:
    print('\nunexpected files (duplicates from a second upload?):', sorted(extra))

assert not missing, (f'missing: {sorted(missing)} -- run this cell again '
                     'and select all six')
print('\nall six uploaded')

Select all six files in one go:
    config.py  folds.json  imbalance.py  models.py  run_grid.py  train_cnn.py



Saving config.py to config.py
Saving folds.json to folds.json
Saving imbalance.py to imbalance.py
Saving models.py to models.py
Saving run_grid.py to run_grid.py
Saving train_cnn.py to train_cnn.py

  config.py            4,852 bytes
  folds.json             751 bytes
  imbalance.py         9,160 bytes
  models.py            8,337 bytes
  run_grid.py         14,237 bytes
  train_cnn.py         4,493 bytes

all six uploaded


In [ ]:
import sys
import numpy as np

sys.path.insert(0, 'src')
import config as C
from imbalance import class_weights, STRATEGIES

y = np.load('artifacts/ds1.npz')['y']
counts = np.bincount(y, minlength=5)
w = class_weights(y)

print('beats      :', f'{len(y):,}', '  (expect 50,977)')
print('classes    :', dict(zip(C.CLASS_NAMES, counts.tolist())))
print('eval on    :', C.EVAL_CLASSES, '  (Q excluded)')
print('strategies :', STRATEGIES)
print()
print('class weights:', {c: round(float(v), 5) for c, v in zip(C.CLASS_NAMES, w)})
print('Q weight     :', float(w[4]), '  <-- MUST be exactly 0.0')

assert len(y) == 50977, 'wrong ds1.npz'
assert 'smote' in STRATEGIES, "OLD imbalance.py -- 'smote' missing. Tell Rishabh."
assert float(w[4]) == 0.0,    'OLD imbalance.py -- Q still weighted. Tell Rishabh.'

# The frozen fold assignment. Fold 0 must hold 10,467 validation beats --
# that is the laptop's split, the one the LR/XGBoost results already use.
import json

with open('src/folds.json') as fh:
    spec = json.load(fh)

pf = {str(k): int(v) for k, v in spec['patient_fold'].items()}
pid = np.load('artifacts/ds1.npz')['pid']
assign = np.array([pf[str(p)] for p in pid])
sizes = [int((assign == f).sum()) for f in range(spec['n_folds'])]

print()
print('folds.json  :', spec['n_patients'], 'patients,', spec['n_folds'], 'folds')
print('made with   : sklearn', spec['generated_with']['sklearn'])
print('fold sizes  :', sizes, '  <-- fold 0 must be 10,467')

assert spec['n_beats'] == len(y), 'folds.json does not match this ds1.npz'
assert sizes[0] == 10467, (
    f'fold 0 has {sizes[0]} beats, expected 10,467 -- wrong folds.json')

print('\nOK -- fixed code and frozen folds confirmed')

beats      : 50,977   (expect 50,977)
classes    : {'N': 45824, 'S': 943, 'V': 3788, 'F': 414, 'Q': 8}
eval on    : ['N', 'S', 'V', 'F']   (Q excluded)
strategies : ['none', 'class_weight', 'focal', 'adasyn', 'smote']

class weights: {'N': 0.0232, 'S': 1.12762, 'V': 0.28071, 'F': 2.56846, 'Q': 0.0}
Q weight     : 0.0   <-- MUST be exactly 0.0

folds.json  : 22 patients, 5 folds
made with   : sklearn 1.9.1
fold sizes  : [10467, 9261, 12606, 9419, 9224]   <-- fold 0 must be 10,467

OK -- fixed code and frozen folds confirmed


In [ ]:
!python src/run_grid.py --models cnn --strategies smote --rr 1 --subset 5000 --epochs 5

DS1: 5,000 beats from 22 patients
grid: 1 models x 1 strategies x 1 rr x 5 folds

SMOKE TEST: folds computed on the fly, results are not usable
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
cnn  smote        rr   macroF1 0.4977 +/- 0.0412   S=0.230 V=0.744 F=0.041   [20s]

Wrote 1 configurations to artifacts/grid_summary.json
Per-fold predictions in artifacts/preds/ — Stages 3-5 read these.


In [ ]:
# Smoke-test predictions use the same filenames as real ones, so they are
# deleted immediately. run_grid.py also validates cached folds against the
# current split, but two guards are better than one here.
!rm -rf {WORK}/artifacts/preds {WORK}/artifacts/grid_summary.json
print('smoke-test output cleared')

smoke-test output cleared


In [ ]:
import glob
import os
import shutil

PREDS_DRIVE = f'{MYDRIVE}/ecg-paper/artifacts/preds'
PREDS_WORK  = f'{WORK}/artifacts/preds'
MARKER      = f'{PREDS_DRIVE}/.qfix_v2'

os.makedirs(PREDS_DRIVE, exist_ok=True)
os.makedirs(PREDS_WORK, exist_ok=True)

if os.path.exists(MARKER):
    done = sorted(glob.glob(f'{PREDS_DRIVE}/cnn_*'))
    for f in done:
        shutil.copy2(f, PREDS_WORK)
    print(f'RESUMING -- restored {len(done)} completed CNN folds')
    print(f'           {50 - len(done)} still to compute')
else:
    stale = glob.glob(f'{PREDS_DRIVE}/cnn_*')
    for f in stale:
        os.remove(f)
    open(MARKER, 'w').close()
    print(f'FIRST RUN -- removed {len(stale)} pre-fix CNN folds from Drive')
    print('           all 50 folds will be computed')

RESUMING -- restored 50 completed CNN folds
           0 still to compute


In [ ]:
!python src/run_grid.py --models cnn

DS1: 50,977 beats from 22 patients
grid: 1 models x 5 strategies x 2 rr x 5 folds

folds: /content/work/src/folds.json (frozen, made with sklearn 1.9.1)
      fold 0: cached prediction does not match current split — recomputing
      fold 1: cached prediction does not match current split — recomputing
      fold 2: cached prediction does not match current split — recomputing
      fold 3: cached prediction does not match current split — recomputing
      fold 4: cached prediction does not match current split — recomputing
cnn  none         rr   macroF1 0.4956 +/- 0.0673   S=0.256 V=0.773 F=0.013   [248s]
      fold 0: cached prediction does not match current split — recomputing
      fold 1: cached prediction does not match current split — recomputing
      fold 2: cached prediction does not match current split — recomputing
      fold 3: cached prediction does not match current split — recomputing
      fold 4: cached prediction does not match current split — recomputing
cnn  none    

In [ ]:
!mkdir -p "{PREDS_DRIVE}"
!cp -r {WORK}/artifacts/preds/* "{PREDS_DRIVE}/" 2>/dev/null

n = !ls {WORK}/artifacts/preds | grep ^cnn | wc -l
print(f'{n[0].strip()} CNN folds backed up   (50 when complete)')

50 CNN folds backed up   (50 when complete)


In [ ]:
import json

with open('artifacts/grid_summary.json') as f:
    rows = json.load(f)

by = {(r['strategy'], r['use_rr']): r for r in rows}

print(f"{'strategy':14} {'rr':>2}   {'macro-F1':>17}   {'S':>6} {'V':>6} {'F':>6}")
print('-' * 62)
for r in rows:
    pc = r['per_class_f1']
    print(f"{r['strategy']:14} {int(r['use_rr']):>2}   "
          f"{r['macro_f1_mean']:.4f} +/- {r['macro_f1_std']:.4f}   "
          f"{pc['S']:6.3f} {pc['V']:6.3f} {pc['F']:6.3f}")


def val(r, what):
    return r['macro_f1_mean'] if what == 'macro-F1' else r['per_class_f1']['S']


print('\n' + '=' * 62)
print('RR ABLATION  --  rr=1 must beat rr=0 in every row')
print('=' * 62)

ok = tot = 0
for s in sorted({r['strategy'] for r in rows}):
    hi_r, lo_r = by.get((s, True)), by.get((s, False))
    if not hi_r or not lo_r:
        continue
    for what in ('macro-F1', 'S'):
        hi, lo = val(hi_r, what), val(lo_r, what)
        tot += 1
        ok += hi > lo
        flag = 'pass' if hi > lo else '*** FAIL ***'
        print(f'  {s:14} {what:9} {hi:.4f}  vs  {lo:.4f}   {flag}')

print(f'\n  {ok}/{tot} comparisons passed', '' if ok == tot else '  <-- REPORT THIS')

print('\n' + '=' * 62)
print('THE THREE NUMBERS TO REPORT  (rr=1, S-class F1)')
print('=' * 62)
for s in ('class_weight', 'adasyn', 'smote'):
    r = by.get((s, True))
    if r:
        print(f'  cnn {s:14} S = {r["per_class_f1"]["S"]:.3f}')
base = by.get(('none', True))
if base:
    print(f'  (baseline  none         S = {base["per_class_f1"]["S"]:.3f})')

strategy       rr            macro-F1        S      V      F
--------------------------------------------------------------
none            1   0.4956 +/- 0.0673    0.256  0.773  0.013
none            0   0.4336 +/- 0.0535    0.073  0.658  0.014
class_weight    1   0.5827 +/- 0.1118    0.346  0.810  0.183
class_weight    0   0.4165 +/- 0.0761    0.064  0.636  0.083
focal           1   0.5673 +/- 0.1135    0.326  0.781  0.180
focal           0   0.4154 +/- 0.0752    0.092  0.723  0.071
adasyn          1   0.4970 +/- 0.0597    0.238  0.740  0.071
adasyn          0   0.4160 +/- 0.0727    0.037  0.615  0.074
smote           1   0.5104 +/- 0.0656    0.257  0.753  0.004
smote           0   0.4166 +/- 0.0638    0.026  0.705  0.081

RR ABLATION  --  rr=1 must beat rr=0 in every row
  adasyn         macro-F1  0.4970  vs  0.4160   pass
  adasyn         S         0.2376  vs  0.0369   pass
  class_weight   macro-F1  0.5827  vs  0.4165   pass
  class_weight   S         0.3457  vs  0.0638   pass
  f

In [ ]:
# Did resampling actually happen? Each prediction records what the sampler did.
import glob
import json
import numpy as np

notes = sorted({json.loads(np.load(p)['meta'].item())['resample']
                for p in glob.glob('artifacts/preds/cnn_*')})
print('resample notes:', notes)

bad = [n for n in notes if 'failed' in n or 'skipped_no_eligible' in n]
print('\nPROBLEM -- report these:', bad) if bad else print('\nno sampler failures')

resample notes: ['adasyn', 'none', 'smote']

no sampler failures


In [ ]:
import glob
import os
import zipfile

OUT = '/content/partB_preds.zip'
folds = sorted(glob.glob(f'{WORK}/artifacts/preds/cnn_*'))

with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in folds:
        z.write(f, 'preds/' + os.path.basename(f))

print(f'{len(folds)} CNN folds packed   (expect 50)')
print(f'{os.path.getsize(OUT) / 1e6:.1f} MB -> {OUT}')

if len(folds) != 50:
    print('\nWARNING: not 50 folds. Finish the run before sending this.')

50 CNN folds packed   (expect 50)
9.7 MB -> /content/partB_preds.zip


In [ ]:
from google.colab import files
files.download('/content/partB_preds.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>